# 01 — Training GALERIA (satu notebook lengkap)

> ## MODEL v1 (11 kelas, 5.659 gambar) -- siap di-run ulang kapan saja
>
> Dataset asli (`data/raw/` versi 5.659 gambar, split CSV train/val/test)
> SUDAH ditimpa dataset yang lebih besar (33.937 gambar) -- lihat
> `notebooks/README.md` bagian "Insiden". Supaya notebook ini tetap bisa
> di-run ulang & menghasilkan ulang run v1 dgn SPLIT YANG SAMA PERSIS, sel
> setelah ini merekonstruksi 3 file split dari 5.659 nama file yang masih
> tersimpan di `data/cache/v1_11class_backup/catalog_embeddings.npz` --
> SUDAH diverifikasi hasilnya cocok 100% dgn checkpoint asli (11 kelas yang
> sama, jumlah train/val/test 3.918/840/840 persis sesuai catatan CLAUDE.md).
>
> Checkpoint/output/export hasil run ulang ditulis ke
> `checkpoints/v1_11class_backup/`, `outputs/v1_11class_backup/`,
> `export/v1_11class_backup/` -- **TIDAK** menyentuh model live di root
> maupun `checkpoints/continued_training/` (tujuan training v2 lanjutan).
> Hyperparameter sudah diisi LANGSUNG dari hasil tuning Optuna sebelumnya
> (`RUN_TUNING = False` by default) -- tidak perlu tuning ulang, cuma
> training final yang direproduksi.

## 1. Setup

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

# pindah ke folder ml/ supaya `import src` & path config jalan
_p = Path.cwd()
while not ((_p / "src").is_dir() and (_p / "configs").is_dir()):
    if _p == _p.parent:
        raise RuntimeError("folder ml/ (berisi src/ & configs/) tidak ketemu")
    _p = _p.parent
os.chdir(_p)
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))

import json, math, time, copy
import numpy as np
import pandas as pd
import torch, torch.nn as nn
import matplotlib.pyplot as plt
import torchvision

torch.backends.cudnn.benchmark = True   # ukuran input tetap -> cuDNN pilih kernel tercepat

print("working dir :", os.getcwd())
print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__, "| torchvision", torchvision.__version__)
print("CUDA        :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 1b. Rekonstruksi split dataset v1 (5.659 gambar, 11 kelas)

`data/raw/*_split.csv` sekarang berisi split utk 33.937 gambar (dataset
sudah diperluas) -- bukan punya v1. Sel di bawah membangun ulang split v1
dari `metadata.csv` (masih berisi SEMUA gambar termasuk yang lama) difilter
ke 5.659 nama file yang tercatat di `data/cache/v1_11class_backup/
catalog_embeddings.npz`, pakai LOGIKA SPLIT YANG SAMA PERSIS (stratified,
`random_state=42`, exclude kelas <40 sampel) seperti saat run asli.

Ditulis ke file BARU (`v1_11class_*_split.csv`), **TIDAK** menimpa
`data/raw/{train,val,test}_split.csv` yang sekarang dipakai training v2.

In [ ]:
from sklearn.model_selection import train_test_split

V1_DIR = Path("data/raw")
V1_FILES = ["v1_11class_train_split.csv", "v1_11class_val_split.csv", "v1_11class_test_split.csv"]

if not all((V1_DIR / f).exists() for f in V1_FILES):
    cat_v1 = np.load(resolve_path("data/cache/v1_11class_backup/catalog_embeddings.npz"))
    v1_filenames = set(cat_v1["filenames"].tolist())
    print(f"nama file v1 (dari catalog_embeddings.npz): {len(v1_filenames)}")

    meta = pd.read_csv(resolve_path("data/raw/metadata.csv"))
    f = meta[meta["filename"].isin(v1_filenames)].copy()
    assert len(f) == len(v1_filenames), "subset filename tidak lengkap -- metadata.csv berubah?"

    MIN_SAMPLES = 40  # sama persis dgn aturan asli (CLAUDE.md: kelas minoritas <40 sampel di-exclude)
    sc = f["style_name"].value_counts()
    valid = sc[sc >= MIN_SAMPLES].index
    f2 = f[f["style_name"].isin(valid)].copy()
    print(f"setelah exclude <{MIN_SAMPLES} sampel: {len(f2)} gambar / {f2['style_name'].nunique()} kelas")

    train_df, temp = train_test_split(f2, test_size=0.3, stratify=f2["style_name"], random_state=42)
    val_df, test_df = train_test_split(temp, test_size=0.5, stratify=temp["style_name"], random_state=42)
    for name, d in zip(("train", "val", "test"), (train_df, val_df, test_df)):
        d.to_csv(V1_DIR / f"v1_11class_{name}_split.csv", index=False)
        print(f"v1_11class_{name}_split.csv: {len(d)} baris")

    # verifikasi ketat -- kalau ini gagal, JANGAN lanjut training (hasil tidak akan sebanding v1 asli)
    ck_check = torch.load(resolve_path("checkpoints/v1_11class_backup/last.pth"),
                          map_location="cpu", weights_only=False)
    assert sorted(f2["style_name"].unique()) == sorted(ck_check["label_to_idx"].keys()), \
        "kelas hasil rekonstruksi TIDAK cocok dgn checkpoint asli!"
    assert (len(train_df), len(val_df), len(test_df)) == (3918, 840, 840), \
        "jumlah split TIDAK cocok dgn catatan CLAUDE.md (3918/840/840)!"
    print("\nVERIFIKASI OK -- split cocok persis dgn run v1 asli.")
else:
    print("split v1 sudah ada, tidak di-generate ulang:", V1_FILES)

In [ ]:
from src.utils import load_config, set_seed, get_device, resolve_path
from src.dataset import build_label_mapping, create_dataloaders
from src.model import build_model, save_checkpoint, load_checkpoint
from src.train import (build_optimizer, build_scheduler, train_one_epoch,
                       validate, make_scaler)
from src.evaluate import run_inference, compute_metrics

cfg = load_config("configs/config.yaml")
cfg["train"]["num_workers"] = 0            # Windows + Jupyter: 0 paling aman
cfg["train"]["early_stopping_patience"] = 6   # val plateau cepat -> patience 10 buang ~10 epoch
# cfg["train"]["epochs"] = 8               # <- uncomment untuk uji cepat

# --- OVERRIDE supaya notebook ini SELALU reproduksi v1 (11 kelas, 5.659
# gambar), TIDAK ikut config.yaml yang sekarang diarahkan ke training v2
# lanjutan (continued_training/, dataset 33.937 gambar). ---
cfg["data"]["train_csv"] = "data/raw/v1_11class_train_split.csv"
cfg["data"]["val_csv"] = "data/raw/v1_11class_val_split.csv"
cfg["data"]["test_csv"] = "data/raw/v1_11class_test_split.csv"
# PENTING: build_label_mapping() cache ke {cache_dir}/label_to_idx__style_name.json
# dan LANGSUNG PAKAI file itu kalau sudah ada -- tanpa override ini, dia akan
# diam-diam pakai cache ROOT (punya model live, 22 kelas!) walau train_csv di
# atas sudah benar. Sudah diverifikasi: tanpa baris ini num_classes jadi 22
# (SALAH), dengan baris ini jadi 11 (benar).
cfg["data"]["cache_dir"] = "data/cache/v1_11class_backup"
cfg["checkpoint"]["dir"] = "checkpoints/v1_11class_backup"
cfg["output"]["dir"] = "outputs/v1_11class_backup"
cfg["output"]["log_file"] = "outputs/v1_11class_backup/train.log"
cfg["output"]["history_plot_path"] = "outputs/v1_11class_backup/training_history.png"
cfg["output"]["history_json_path"] = "outputs/v1_11class_backup/history.json"
cfg["output"]["confusion_matrix_path"] = "outputs/v1_11class_backup/confusion_matrix.png"
cfg["output"]["metrics_report_path"] = "outputs/v1_11class_backup/metrics_report.json"
cfg["embedding"]["output_path"] = "data/cache/v1_11class_backup/catalog_embeddings_rerun.npz"
cfg["embedding"]["faiss_index_path"] = "data/cache/v1_11class_backup/catalog_rerun.faiss"
cfg["export"]["checkpoint"] = "checkpoints/v1_11class_backup/best.pth"
cfg["export"]["torchscript_path"] = "export/v1_11class_backup/model.torchscript"
cfg["export"]["onnx_path"] = "export/v1_11class_backup/model.onnx"

# Hyperparameter LANGSUNG dari hasil Optuna tuning sebelumnya -- tidak perlu
# re-tuning (lihat sel RUN_TUNING di bawah, default False, else-branch
# otomatis mengisi nilai yang sama ini).
TUNED_PARAMS = {
    "learning_rate": 0.0002584949196006611,
    "backbone_lr_mult": 0.18427033274320545,
    "weight_decay": 0.0009849813063176139,
    "dropout": 0.2017262123368978,
    "unfreeze_frac": 0.4406353983164585,
}

set_seed(cfg["seed"])
device = get_device(cfg["device"])
use_amp = bool(cfg["train"]["mixed_precision"]) and device.type == "cuda"
print("device:", device, "\n")

# variabel BERSAMA (dipakai lintas sel; taruh di sini biar re-run sel individual aman)
MEAN = torch.tensor(cfg["image"]["mean"]).view(3, 1, 1)
STD  = torch.tensor(cfg["image"]["std"]).view(3, 1, 1)

def denorm(t):
    return (t * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

for sec in ["image", "augmentation", "model", "train", "checkpoint", "eval"]:
    print(f"[{sec}]"); print(json.dumps(cfg[sec], indent=2, ensure_ascii=False)); print()

## 2. Data

In [ ]:
label_to_idx = build_label_mapping(cfg)
idx_to_label = {v: k for k, v in label_to_idx.items()}
num_classes  = len(label_to_idx)
class_names  = [idx_to_label[i] for i in range(num_classes)]   # bersama
print(f"{num_classes} kelas:", class_names)

dist = {}
for name in ("train", "val", "test"):
    d = pd.read_csv(resolve_path(cfg["data"][f"{name}_csv"]))
    dist[name] = d[cfg["data"]["label_column"]].value_counts()
dist = pd.DataFrame(dist).fillna(0).astype(int).sort_values("train", ascending=False)
print(dist)

ax = dist.plot(kind="bar", figsize=(13, 4))
ax.set_title(f"Distribusi kelas per split — imbalance train "
             f"{dist['train'].max()/dist['train'].min():.1f}x")
ax.set_ylabel("jumlah gambar"); plt.tight_layout(); plt.show()

In [ ]:
loaders = create_dataloaders(cfg)
print({k: f"{len(v)} batch / {len(v.dataset)} img" for k, v in loaders.items()})

In [ ]:
# contoh 1 batch train (SquarePad sudah di-preprocess; ini + augmentasi TrivialAugment)
imgs, labels = next(iter(loaders["train"]))
n = min(12, len(imgs))
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for i, ax in enumerate(axes.flat):
    if i < n:
        ax.imshow(denorm(imgs[i])); ax.set_title(idx_to_label[labels[i].item()], fontsize=8)
    ax.axis("off")
fig.suptitle("1 batch train setelah augmentasi"); plt.tight_layout(); plt.show()

## 3. Hyperparameter Tuning (Optuna) — opsional

Cari `learning_rate`, `backbone_lr_mult`, `weight_decay`, `dropout`,
`unfreeze_frac` terbaik lewat pencarian otomatis (TPE sampler + pruning),
bukan tebakan manual. Tiap trial = training singkat (`TUNE_EPOCHS`, bukan
penuh) — cukup untuk mengurutkan konfigurasi mana yang lebih baik, karena val
macro-F1 kita sudah kelihatan plateau cepat (~epoch 10).

**Set `RUN_TUNING = False` setelah dapat hyperparameter terbaik**, supaya
"Run All" berikutnya tidak mengulang pencarian yang mahal — cfg yang sudah
diupdate otomatis dipakai sel Training di bawah.

Progress disimpan ke `outputs/tune_study.db` (SQLite) — **resumable**: kalau
notebook/kernel terhenti di tengah jalan, jalankan ulang sel ini, trial yang
sudah selesai tidak diulang.

In [ ]:
RUN_TUNING  = False   # default False -- hyperparameter sudah diketahui (TUNED_PARAMS di atas),
                      # ganti True kalau mau re-verifikasi/re-cari dari nol (makan waktu)
N_TRIALS    = 15
TUNE_EPOCHS = 8        # budget per trial (val kita plateau ~epoch 10 di run penuh)

In [ ]:
if RUN_TUNING:
    import optuna
    from optuna.pruners import MedianPruner
    from optuna.samplers import TPESampler

    def objective(trial):
        t_cfg = copy.deepcopy(cfg)
        t_cfg["train"]["learning_rate"]    = trial.suggest_float("learning_rate", 1e-4, 1e-3, log=True)
        t_cfg["train"]["backbone_lr_mult"] = trial.suggest_float("backbone_lr_mult", 0.02, 0.5, log=True)
        t_cfg["train"]["weight_decay"]     = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
        t_cfg["model"]["dropout"]          = trial.suggest_float("dropout", 0.2, 0.6)
        t_cfg["model"]["unfreeze_frac"]    = trial.suggest_float("unfreeze_frac", 0.15, 0.6)
        t_cfg["train"]["epochs"] = TUNE_EPOCHS   # cosine schedule disesuaikan budget trial

        set_seed(cfg["seed"])
        t_model = build_model(t_cfg, num_classes=num_classes).to(device)
        t_crit  = nn.CrossEntropyLoss(label_smoothing=t_cfg["train"]["label_smoothing"])
        t_opt   = build_optimizer(t_cfg, t_model)
        t_sched = build_scheduler(t_cfg, t_opt)
        t_scaler = make_scaler(device.type, use_amp)

        best_f1 = -1.0
        for ep in range(1, TUNE_EPOCHS + 1):
            train_one_epoch(t_model, loaders["train"], t_crit, t_opt, device, t_scaler,
                            t_cfg["train"].get("grad_clip_norm"))
            va = validate(t_model, loaders["val"], t_crit, device)
            if t_sched is not None:
                t_sched.step()
            best_f1 = max(best_f1, va["macro_f1"])
            trial.report(va["macro_f1"], ep)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return best_f1

    study = optuna.create_study(
        study_name="galeria_v1_11class",   # study TERPISAH dari v2 (nama beda), tidak saling ganggu
        storage=f"sqlite:///{resolve_path('outputs/v1_11class_backup/tune_study.db')}",
        load_if_exists=True,
        direction="maximize",
        sampler=TPESampler(seed=cfg["seed"]),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    )
    n_done = sum(t.state.is_finished() for t in study.trials)
    print(f"trial sudah selesai sebelumnya: {n_done} (resume otomatis)")
    if n_done < N_TRIALS:
        study.optimize(objective, n_trials=N_TRIALS - n_done, show_progress_bar=True)

    print(f"\nbest trial #{study.best_trial.number} | val macro-F1 = {study.best_value:.4f}")
    print("best params:", json.dumps(study.best_params, indent=2))

    study.trials_dataframe().to_csv(resolve_path("outputs/v1_11class_backup/tune_trials.csv"), index=False)
    resolve_path("outputs/v1_11class_backup/tune_best_params.json").write_text(
        json.dumps(study.best_params, indent=2), encoding="utf-8")

    # terapkan hyperparameter terbaik ke cfg -> dipakai sel Model & Training di bawah
    cfg["train"]["learning_rate"]    = study.best_params["learning_rate"]
    cfg["train"]["backbone_lr_mult"] = study.best_params["backbone_lr_mult"]
    cfg["train"]["weight_decay"]     = study.best_params["weight_decay"]
    cfg["model"]["dropout"]          = study.best_params["dropout"]
    cfg["model"]["unfreeze_frac"]    = study.best_params["unfreeze_frac"]
    print("\ncfg diupdate dengan hyperparameter terbaik (baru) -> Training di bawah otomatis memakainya")
else:
    # TERAPKAN hyperparameter yang SUDAH diketahui dari tuning v1 asli (TUNED_PARAMS,
    # didefinisikan di sel config) -- BUKAN "pakai default config.yaml" seperti versi
    # lama (itu bug: default config.yaml = nilai BELUM di-tuning, beda dari yang
    # sebenarnya dipakai run v1 asli).
    cfg["train"]["learning_rate"]    = TUNED_PARAMS["learning_rate"]
    cfg["train"]["backbone_lr_mult"] = TUNED_PARAMS["backbone_lr_mult"]
    cfg["train"]["weight_decay"]     = TUNED_PARAMS["weight_decay"]
    cfg["model"]["dropout"]          = TUNED_PARAMS["dropout"]
    cfg["model"]["unfreeze_frac"]    = TUNED_PARAMS["unfreeze_frac"]
    print("RUN_TUNING=False -> pakai TUNED_PARAMS (hasil tuning v1 asli), BUKAN default config.yaml:")
    print(json.dumps(TUNED_PARAMS, indent=2))

In [ ]:
if RUN_TUNING:
    try:
        import optuna.visualization.matplotlib as ovm
        ovm.plot_optimization_history(study); plt.tight_layout(); plt.show()
        ovm.plot_param_importances(study); plt.tight_layout(); plt.show()
    except Exception as e:
        print("plot Optuna dilewati:", e)

    top = (study.trials_dataframe()
           .query("state == 'COMPLETE'")
           .sort_values("value", ascending=False)
           [["number", "value", "params_learning_rate", "params_backbone_lr_mult",
             "params_weight_decay", "params_dropout", "params_unfreeze_frac"]])
    print(top.head(10).to_string(index=False))

## 4. Model

In [ ]:
model = build_model(cfg, num_classes=num_classes).to(device)
n_tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in model.parameters())
_uf = (f"unfreeze_last_n_blocks={cfg['model']['unfreeze_last_n_blocks']}"
       if model.arch.startswith("resnet")
       else f"unfreeze_frac={cfg['model'].get('unfreeze_frac', 0.3)}")
print(f"{model.arch} | emb_dim {model.embedding_dim} | "
      f"trainable {n_tr/1e6:.2f}M / {n_all/1e6:.2f}M  ({cfg['model']['train_mode']}, {_uf})")
print("(hyperparameter dari tuning baru)" if RUN_TUNING else "(hyperparameter dari TUNED_PARAMS -- hasil tuning v1 asli)")

## 5. Training

Loop pakai fungsi dari `src.train`. Tiap epoch dicatat: loss, **MSE (Brier)**,
accuracy, macro-F1 (train & val), learning rate. Loss tiap batch -> `step_losses`.

**EMA** (exponential moving average) bobot di-update tiap epoch -> checkpoint
lebih stabil saat val macro-F1 plateau/berisik. Di akhir: dipilih yang terbaik
antara `best.pth` (per-epoch) vs EMA, di val.

> Loop ini SENGAJA ditulis inline (bukan panggil `src.train.main`) supaya
> terlihat & bisa dimodifikasi. Konsekuensi: sedikit duplikasi dengan
> `src/train.py:main()` — kalau ubah salah satu, samakan yang lain.

In [ ]:
class EMA:
    # rata-rata bergerak eksponensial dari state_dict model
    def __init__(self, model, decay=0.8):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            s = self.shadow[k]
            if v.dtype.is_floating_point:
                s.mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                s.copy_(v)

    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)


criterion = nn.CrossEntropyLoss(label_smoothing=cfg["train"]["label_smoothing"])
optimizer = build_optimizer(cfg, model)
scheduler = build_scheduler(cfg, optimizer)
scaler    = make_scaler(device.type, use_amp)
ema       = EMA(model, decay=0.8)

monitor     = cfg["checkpoint"]["monitor"]
monitor_key = monitor.removeprefix("val_")
grad_clip   = cfg["train"].get("grad_clip_norm") or None
patience    = cfg["train"]["early_stopping_patience"]
ckpt_dir    = resolve_path(cfg["checkpoint"]["dir"])

history = {k: [] for k in ["train_loss", "val_loss", "train_mse", "val_mse",
                           "train_accuracy", "val_accuracy",
                           "train_macro_f1", "val_macro_f1", "lr"]}
step_losses = []
best, no_improve = -math.inf, 0
t0 = time.time()

for epoch in range(1, cfg["train"]["epochs"] + 1):
    tr = train_one_epoch(model, loaders["train"], criterion, optimizer, device,
                         scaler, grad_clip, step_losses=step_losses)
    ema.update(model)
    va = validate(model, loaders["val"], criterion, device)
    lr_now = optimizer.param_groups[0]["lr"]
    if scheduler is not None:
        scheduler.step()

    for k in ("loss", "mse", "accuracy", "macro_f1"):
        history[f"train_{k}"].append(tr[k]); history[f"val_{k}"].append(va[k])
    history["lr"].append(lr_now)

    print(f"epoch {epoch:2d}/{cfg['train']['epochs']} | lr {lr_now:.1e} | "
          f"train loss {tr['loss']:.3f} mse {tr['mse']:.3f} f1 {tr['macro_f1']:.3f} | "
          f"val loss {va['loss']:.3f} mse {va['mse']:.3f} f1 {va['macro_f1']:.3f} "
          f"acc {va['accuracy']:.3f}")

    if va[monitor_key] > best:
        best, no_improve = va[monitor_key], 0
        save_checkpoint(model, ckpt_dir / "best.pth", epoch=epoch, monitor=monitor,
                        monitor_value=best, val_metrics=va,
                        label_to_idx=label_to_idx, config=cfg)
        print(f"   -> best {monitor} = {best:.4f}  (checkpoints/best.pth)")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping — {monitor} tidak membaik {patience} epoch."); break

save_checkpoint(model, ckpt_dir / "last.pth", epoch=epoch, val_metrics=va,
                label_to_idx=label_to_idx, config=cfg)

# --- pilih model final: best.pth vs EMA (di val) ---
ema_model = build_model(cfg, num_classes=num_classes).to(device)
ema.copy_to(ema_model)
ema_va = validate(ema_model, loaders["val"], criterion, device)
best_va = load_checkpoint(build_model(cfg, num_classes).to(device),
                          ckpt_dir / "best.pth", map_location=str(device))["val_metrics"]
print(f"\nval macro-F1  best.pth {best_va['macro_f1']:.4f}  vs  EMA {ema_va['macro_f1']:.4f}")
if ema_va["macro_f1"] > best_va["macro_f1"]:
    save_checkpoint(ema_model, ckpt_dir / "best.pth", epoch=epoch, monitor=monitor,
                    monitor_value=ema_va["macro_f1"], val_metrics=ema_va,
                    label_to_idx=label_to_idx, config=cfg, note="EMA")
    print("   -> EMA menang, best.pth di-update ke bobot EMA")

# flag overfitting
gap = history["train_macro_f1"][-1] - history["val_macro_f1"][-1]
if gap > 0.15:
    print(f"\n[!] OVERFIT: gap train-val macro-F1 = {gap:.2f} (train {history['train_macro_f1'][-1]:.2f} "
          f"vs val {history['val_macro_f1'][-1]:.2f}). Lever utama: tambah data.")

resolve_path(cfg["output"]["history_json_path"]).write_text(
    json.dumps({**history, "step_losses": step_losses}, indent=2), encoding="utf-8")
print(f"\ntotal {(time.time()-t0)/60:.1f} menit")

## 6. Visualisasi Training

### 6a. Loss per step — bukti gradient descent

In [ ]:
plt.figure(figsize=(13, 4))
plt.plot(step_losses, lw=0.5, alpha=0.35, color="gray", label="loss per batch")
k = max(1, len(step_losses) // 60)
if len(step_losses) > k:
    ma = np.convolve(step_losses, np.ones(k) / k, mode="valid")
    plt.plot(range(k - 1, len(step_losses)), ma, lw=2, color="tab:red", label=f"moving avg ({k})")
plt.xlabel("training step (batch)"); plt.ylabel("CrossEntropy loss")
plt.title(f"Loss turun sepanjang {len(step_losses)} step optimisasi -> gradient descent bekerja")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(resolve_path("outputs/v1_11class_backup/train_step_loss.png"), dpi=110); plt.show()
print(f"loss step 1 : {step_losses[0]:.3f}  ->  step {len(step_losses)} : {step_losses[-1]:.3f}"
      f"  (turun {step_losses[0]-step_losses[-1]:.3f})")

### 6b. Learning curve — train vs val (loss, MSE/Brier, macro-F1)

In [ ]:
ep = range(1, len(history["train_loss"]) + 1)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].plot(ep, history["train_loss"], "o-", ms=3, label="train")
ax[0].plot(ep, history["val_loss"],  "s-", ms=3, label="val")
ax[0].set_title("Loss (CrossEntropy)"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(ep, history["train_mse"], "o-", ms=3, label="train")
ax[1].plot(ep, history["val_mse"],  "s-", ms=3, label="val")
ax[1].set_title("MSE / Brier = ||prob - onehot||^2"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=0.3)
ax[2].plot(ep, history["train_macro_f1"], "o-", ms=3, label="train")
ax[2].plot(ep, history["val_macro_f1"],  "s-", ms=3, label="val")
ax[2].set_title("macro-F1"); ax[2].set_xlabel("epoch"); ax[2].set_ylim(0, 1); ax[2].legend(); ax[2].grid(alpha=0.3)
gap = history["train_macro_f1"][-1] - history["val_macro_f1"][-1]
ax[2].annotate(f"gap train-val = {gap:.2f}\n(lebar -> overfit)",
               xy=(list(ep)[-1], history["val_macro_f1"][-1]),
               xytext=(0.3, 0.12), textcoords="axes fraction", arrowprops=dict(arrowstyle="->"))
fig.suptitle("Learning curve — train (bulat) vs val (kotak)", fontsize=12)
plt.tight_layout(); plt.savefig(resolve_path("outputs/v1_11class_backup/train_learning_curve.png"), dpi=110); plt.show()

### 6c. Jadwal learning rate

In [ ]:
ep = range(1, len(history["lr"]) + 1)
plt.figure(figsize=(9, 3.5))
plt.plot(ep, history["lr"], "o-", ms=3, color="tab:green")
plt.xlabel("epoch"); plt.ylabel("learning rate"); plt.title(f"LR schedule ({cfg['train']['scheduler']})")
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(resolve_path("outputs/v1_11class_backup/train_lr_schedule.png"), dpi=110); plt.show()

## 7. Evaluasi Test Set (checkpoint terpilih)

In [ ]:
best_model = build_model(cfg, num_classes=num_classes).to(device)
ck = load_checkpoint(best_model, ckpt_dir / "best.pth", map_location=str(device))
print(f"best.pth: epoch {ck.get('epoch')} | {ck.get('monitor')} = {ck.get('monitor_value'):.4f}"
      f" | {ck.get('note', 'per-epoch')}")

# perbandingan best vs last di test (FYI)
last_model = build_model(cfg, num_classes=num_classes).to(device)
load_checkpoint(last_model, ckpt_dir / "last.pth", map_location=str(device))
ev = cfg.get("eval", {})
for tag, mdl in [("best", best_model), ("last", last_model)]:
    yt, yp, _ = run_inference(mdl, loaders["test"], device, tta=ev.get("tta", False))
    from sklearn.metrics import f1_score, accuracy_score
    print(f"  {tag:4s}: test acc {accuracy_score(yt, yp):.3f} | macro-F1 {f1_score(yt, yp, average='macro'):.3f}")

In [ ]:
y_true, y_pred, y_prob = run_inference(best_model, loaders["test"], device, tta=ev.get("tta", False))
metrics = compute_metrics(y_true, y_pred, y_prob, idx_to_label,
                          subset_min_support=ev.get("subset_min_support", 8))

print(f"TEST (n={metrics['n_test']}, tta={ev.get('tta', False)})")
for k, lab in [("accuracy", "accuracy      "), ("top3_accuracy", "top-3 accuracy"),
               ("macro_f1", "macro-F1      "), ("weighted_f1", "weighted-F1   "),
               ("mse", "MSE / Brier   ")]:
    print(f"  {lab}: {metrics[k]:.3f}")

print("\nkonteks:")
print("  - baseline (1.132 gambar, 12 kelas)   : test macro-F1 ~0.44")
print("  - literatur WikiArt style (data penuh): ~0.55-0.65 accuracy")
print(f"  - sekarang (5.6k gambar, {num_classes} kelas, tuning={RUN_TUNING}): test macro-F1 {metrics['macro_f1']:.2f}")
print("  - CATATAN: subset 5/72 shard, split TIDAK artist-disjoint, sebagian style")
print("    ter-confound dgn pelukis (Symbolism~Roerich). Jangan overclaim.")

metrics["checkpoint"] = {"epoch": ck.get("epoch"), "monitor_value": ck.get("monitor_value"),
                         "note": ck.get("note", "per-epoch")}
metrics["tuning"] = {"ran": RUN_TUNING,
                     "best_params": (study.best_params if RUN_TUNING else TUNED_PARAMS)}
resolve_path(cfg["output"]["metrics_report_path"]).write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")

### 7a. F1 per kelas

In [ ]:
rep = metrics["per_class"]
df = (pd.DataFrame({k: rep[k] for k in class_names}).T
        [["precision", "recall", "f1-score", "support"]].round(3).sort_values("f1-score"))
print(df)
ax = df["f1-score"].plot(kind="barh", figsize=(9, 4), color="tab:blue")
ax.set_xlim(0, 1); ax.set_xlabel("F1"); ax.set_title("F1 per kelas (test)")
plt.tight_layout(); plt.show()

### 7b. Confusion matrix (ternormalisasi per baris)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(9, 8))
try:
    import seaborn as sns
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax, cbar=True)
except ImportError:
    im = ax.imshow(cm_norm, cmap="Blues")
    ax.set_xticks(range(num_classes), class_names, rotation=90)
    ax.set_yticks(range(num_classes), class_names); fig.colorbar(im, ax=ax)
ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual"); ax.set_title("Confusion matrix (test)")
plt.tight_layout(); plt.savefig(resolve_path(cfg["output"]["confusion_matrix_path"]), dpi=110); plt.show()

### 7c. Contoh prediksi acak (hijau = benar, merah = salah)

In [ ]:
# ambil 15 indeks ACAK dari test set (bukan batch pertama yg bias shard-0)
test_ds = loaders["test"].dataset
rng = np.random.default_rng(cfg["seed"])
pick = rng.choice(len(test_ds), size=15, replace=False)
xs = torch.stack([test_ds[i][0] for i in pick])
ys = [test_ds[i][1] for i in pick]
with torch.no_grad():
    pr = torch.softmax(best_model(xs.to(device)), dim=1).argmax(1).cpu().tolist()

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for ax, xi, t, p in zip(axes.flat, xs, ys, pr):
    ax.imshow(denorm(xi)); ax.axis("off")
    ax.set_title(f"{idx_to_label[p]}\n({idx_to_label[t]})", fontsize=8,
                 color="green" if t == p else "red")
fig.suptitle("prediksi (aktual) — 15 sampel test acak"); plt.tight_layout(); plt.show()

## 8. Catatan interpretasi (untuk laporan)

- **Pretext task** = klasifikasi style. Tujuan akhir = embedding untuk Visual
  Search; ukur retrieval@k terpisah (`src/embedding.py`).
- **Hyperparameter tuning**: `learning_rate`, `backbone_lr_mult`, `weight_decay`,
  `dropout`, `unfreeze_frac` dicari otomatis via Optuna (TPE + median pruning),
  bukan tebakan manual — hasil di `outputs/tune_trials.csv` /
  `tune_best_params.json`, bisa dilampirkan ke laporan sbg bukti metodologi.
- **Overfitting**: gap train-val macro-F1 lebar -> model masih menghafal. Obat
  utama = **tambah data** (shard WikiArt). EMA + patience 6 + tuning mengurangi
  noise, bukan menghilangkan overfit sepenuhnya.
- **F1 per kelas** jangan dibaca polos: kelas yang di data ini didominasi 1
  pelukis (Symbolism~Roerich, Northern_Renaissance~Durer) -> F1 tinggi bisa
  berarti pengenalan pelukis, bukan aliran.
- **Model**: `convnext_small` dipilih di atas base/large karena dataset 5.6k
  terlalu kecil untuk kapasitas ekstra (model besar overfit + VRAM 4 GB +
  embedding lebih gemuk untuk Visual Search).
- Keterbatasan: subset shard 0-4 dari 72, tidak representatif, split tidak
  artist-disjoint. Sebut di semua klaim performa.
- Artefak: `outputs/tune_trials.csv`, `tune_best_params.json`,
  `train_step_loss.png`, `train_learning_curve.png`, `train_lr_schedule.png`,
  `confusion_matrix.png`, `metrics_report.json`, `history.json`.